# 02 - Face detection and overlap removal
**Stage:** find the faces (RetinaFace), drop faces smaller than 32 px, remove overlapping boxes with the paper's
Algorithm 1 (keep the larger face), number the faces left to right, and get a first emotion guess from DeepFace.

| | |
|---|---|
| **Input** | `data/images_proc/*.jpg` (from notebook 01) |
| **Output** | `data/faces_auto.json` (boxes, 5 landmarks, DeepFace guess for every face) |
| **Code used** | `src/detection.py` |

If RetinaFace cannot run, the code falls back to OpenCV YuNet and prints a warning - mention that in your report.

In [ ]:
# Setup: find the repo root so `from src import ...` works (run this notebook from inside the repo).
import sys
from pathlib import Path
ROOT = next((p for p in [Path.cwd(), Path.cwd().parent] if (p / "src").is_dir()), None)
assert ROOT is not None, "Run from inside the repo (the folder containing src/). In Colab: upload the repo zip, unzip it and %cd into it."
sys.path.insert(0, str(ROOT))
from src import config, utils
print("repo root:", ROOT)

In [ ]:
# Colab only: install the detectors (skip if already installed)
!pip -q install deepface retina-face tf-keras tqdm

In [ ]:
import os
os.environ.setdefault("TF_USE_LEGACY_KERAS", "1")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
import pandas as pd
from src import detection

records, skipped = detection.detect_all(config.IMG_DIR)
if skipped:
    print("no faces found in (skipped):", skipped)
utils.save_json(records, config.FACES_JSON)
print(f"{len(records)} images, {sum(r['n_faces'] for r in records)} faces | detector: {detection.DETECTOR['name']}")
print(pd.Series([config.difficulty(r["n_faces"]) for r in records]).value_counts().reindex(config.DIFFICULTIES).fillna(0).astype(int).to_string())

In [ ]:
# Visual check: boxes and landmarks on the first images
import cv2, matplotlib.pyplot as plt
fig, axes = plt.subplots(1, min(3, len(records)), figsize=(18, 6))
for ax, rec in zip(axes if len(records) > 1 else [axes], records[:3]):
    img = cv2.imread(rec["path"])
    for f in rec["faces"]:
        x, y, w, h = f["box"]
        cv2.rectangle(img, (x, y), (x + w, y + h), (0, 200, 0), max(2, img.shape[1] // 400))
        for px, py in f["landmarks"].values():
            cv2.circle(img, (int(px), int(py)), max(2, w // 25), (0, 0, 255), -1)
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); ax.set_title(f"{rec['image']} ({rec['n_faces']} faces)"); ax.axis("off")
plt.tight_layout(); plt.show()

**Next:** `03_labeling.ipynb`